<a href="https://colab.research.google.com/github/Elvis0912/MLProject/blob/main/Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pdfplumber
import pytesseract
from pdf2image import convert_from_path
import os
import tempfile
from dotenv import load_dotenv
import google.generativeai as genai

# Load environment variables
load_dotenv()
genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

# Function to extract text from a PDF
def extract_text_from_pdf(pdf_path: str) -> str:
    text = ""
    try:
        with pdfplumber.open(pdf_path) as pdf:
            for page in pdf.pages:
                page_text = page.extract_text()
                if page_text:
                    text += page_text
        if text.strip():
            return text.strip()
    except Exception as e:
        print(f"Direct text extraction failed: {e}")

    # OCR fallback
    try:
        print("Falling back to OCR...")
        images = convert_from_path(pdf_path)
        for image in images:
            text += pytesseract.image_to_string(image) + "\n"
    except Exception as e:
        print(f"OCR failed: {e}")

    return text.strip()

# Function to analyze resume and return insights
def analyze_resume_from_file(pdf_file, job_description: str = None) -> dict:
    try:
        # Save uploaded file to a temp location
        with tempfile.NamedTemporaryFile(delete=False, suffix=".pdf") as tmp:
            tmp.write(pdf_file)
            tmp_path = tmp.name

        # Extract text from PDF
        resume_text = extract_text_from_pdf(tmp_path)
        os.remove(tmp_path)

        if not resume_text:
            return {"error": "Could not extract any text from the resume."}

        # Prepare prompt
        model = genai.GenerativeModel("gemini-1.5-flash")
        prompt = f"""
        You are an expert HR and Technical Recruiter. Evaluate the following resume and provide:

        - A professional score out of 100 (label it clearly as "Score").
        - Strengths and weaknesses.
        - List of existing skills.
        - Recommended skills to learn or improve.
        - Suggested online courses (name + platform).

        Resume:
        {resume_text}
        """

        if job_description:
            prompt += f"""

            Also compare the resume to this job description:

            {job_description}

            Highlight how well the resume aligns with it.
            """

        response = model.generate_content(prompt)
        return {"analysis": response.text.strip()}

    except Exception as e:
        return {"error": str(e)}